In [2]:
!pip install python-dotenv

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from openai import OpenAI
# client = OpenAI(api_key = os.environ['OPENAI_KEY'])
client = OpenAI()
def ask_llm(prompt, model = 'gpt-5-nano', temp=1):
    response = client.chat.completions.create(
        model=model,    
        temperature=temp,
        messages=[{
            'role':'user',
            'content' : prompt
        }] 
    )       
    return response.choices[0].message.content   

In [ ]:
# 1. zero-shot Prompting
# 예시없이 지시사항만 던지는것 - LLM의 기본기능
prompt = "이 문장의 감정을 분류해: '오늘 점심 메뉴가 품절이라 너무 슬퍼.'"
print(ask_llm(prompt,temp=1))

주요 감정: 부정적  
세부 감정: 슬픔(강한 수준), 실망도 포함될 수 있습니다.


In [8]:
# few-shot Prompting
# 이렇게 하는거야 라고 예시(Shot)를 몇개 보여줘서 성능을 높이는 기술
prompt = """
단어를 이모지로 바꿔줘
사과 -> 🍎
자동차 -> 🚗
고양이 -> 🐱
비행기->
"""
print(ask_llm(prompt,temp=1))

비행기 -> ✈️


In [14]:
# Chain-of-Thought Prompting  Cot(생각의 사슬)
prompt = """
질문 : 5개의 사과중 2개를 먹고 3개를 더 샀어. 몇개 남았지
사과의 단가는 100원
지급한 금액 : 1000원
총 남은 사과의 개수를 세고 그리고 사과를 구입할때 드는 비용을 
계산해서 거스름돈을 계산해줘
계산은 먹은 사과와 남은 사과를 모두 포함한 금액
마지막 출력은 전체 로직을 점검해서 오류가 있는지 확인하고 결과알려줘
"""
print(ask_llm(prompt,temp=1))

다음과 같이 계산해볼게요. 주어진 값은
- 처음 사과: 5개
- 사과를 먹음: 2개
- 남은 사과(먹고 남긴 후) = 5 - 2 = 3개
- 새로 산 사과: 3개
- 산 사과의 단가: 100원
- 지급한 금액: 1000원

1) 현재 남아 있는 사과의 총 개수
- 먹은 뒤 남은 사과 3개 + 새로 산 사과 3개 = 6개

2) 구입에 드는 비용(요청대로 “먹은 사과와 남은 사과를 모두 포함”한 금액으로 계산)
- 총 포함 사과 수 = 먹은 사과 2개 + 현재 남은 사과 6개 = 8개
- 8개 × 100원 = 800원

3) 거스름돈
- 지급액 1000원 − 총 포함 비용 800원 = 200원

추가로 명확히 해두면 좋은 점
- 일반적으로는 현재 가진 사과(6개) 만 계산해서 거스름돈을 구합니다.
  - 이 경우 총 비용 = 6개 × 100원 = 600원
  - 거스름돈 = 1000원 − 600원 = 400원
- eaten(먹은) 부분은 이미 소비된 것이므로, 이번 거래에서 실제로는 지불 대상이 아닙니다. 따라서 “먹은 사과를 포함한다”는 해석은 재무적으로 혼동의 여지가 있습니다.

결론
- 요청대로 먹은 사과를 포함하면: 남은 사과 6개, 총 포함 비용 800원, 거스름돈 200원.
- 일반적인 해석(현재 갖고 있는 6개를 기준)으로 하면: 거스름돈 400원.

로직 점검 결과 및 오류 여부
- 초기 값과 연산은 일관되나, “먹은 사과를 포함한 비용”은 실제 거래 맥락과 다릅니다. 이 해석은 의도했는지 확인이 필요합니다.
- 남은 사과의 수는 6개로 올바르게 계산됩니다.
- 거스름돈 계산은 포함하는 항목에 따라 200원 또는 400원이 나옵니다. 어떤 해석이 맞는지 의도대로 맞춰 정리하는 것이 좋습니다.

원하시면 특정 해석(예: 현재 가진 6개만 계산)으로 다시 정확한 결과를 정리해 드릴게요.
